# Project 1 - Information Retrieval as a recommender system

In [50]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Pre-processing

### Load datas

In [51]:
reviews = pd.read_csv('data/reviews83325.csv')
reviews.head()

/var/folders/jc/0rdgc11d06dgm3sng453_7yr0000gn/T/ipykernel_3491/3645581194.py:1: DtypeWarning: Columns (15,16,17,18,19,20) have mixed types. Specify dtype option on import or set low_memory=False.
  reviews = pd.read_csv('data/reviews83325.csv')


,id,idplace,titre,idauteur,review,note,date_review,date_visit,langue,published_platform,...,subratings,machine_translated,machine_translatable,owner_id,owner_langue,owner_date_review,owner_connection,owner_responder,owner_response,owner_title
0,771569620,188467,February,F645CC9429E8A40EB1F5A487780EC683,Personally I think it is the most beautiful sq...,5,23/9/2020 11:14:11,2020-02,en,Desktop,...,[],False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,769814072,188467,Nice green square,AFFB511F21DF819776CB2F8013034382,We walked through this lovely park but did not...,4,11/9/2020 07:52:32,2019-10,en,Desktop,...,[],False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,758953508,188467,A Treasure in Paris,9262311F3378F8CC4709DD4D92380278,We come back to this huge square every time we...,5,5/7/2020 03:16:32,2019-10,en,Desktop,...,[],False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,755705747,188467,A place to take a breath from exploring Paris.,836BAA8786B81033412B950CF5BDA70C,The most beautiful square in Paris has a stran...,5,31/5/2020 19:36:17,2019-06,en,Desktop,...,[],False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,750396525,188467,Beautiful and elegant,8FEC7B1C9034428EFF3BF8728B036829,"Lovely architecture, looks different again to ...",4,11/3/2020 06:16:38,2020-03,en,Desktop,...,[],False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [52]:
reviews.shape

(340385, 21)

In [53]:
tripadvisor = pd.read_csv('data/Tripadvisor.csv')
tripadvisor.head()

,id,idTrip,fromId,nom,url,rating,nbAvis,nbAvisRecupere,latitude,longitude,...,ap_exclusion,ap_inclusions,ap_introduction,ap_primary_supplier_attraction_id,ap_primary_supplier_subtype,ap_primary_ta_geo_id,ap_product_code,ap_product_highlights,ap_product_text,ap_raw
0,188467,187147.0,187070.0,Place des Vosges,https://www.tripadvisor.fr/Attraction_Review-g...,4.108407,5663,5664.0,48.855614,2.365553,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,188468,187147.0,187070.0,Rue des Francs Bourgeois,https://www.tripadvisor.fr/Attraction_Review-g...,3.316532,73,73.0,48.858140,2.359880,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,188470,187147.0,NaN,Village Saint-Paul,https://www.tripadvisor.fr/Attraction_Review-g...,3.017118,98,98.0,48.853733,2.361295,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,188471,187147.0,187144.0,Au Passe-partout,https://www.tripadvisor.fr/Attraction_Review-g...,2.743157,2,2.0,48.853470,2.361600,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,188472,187147.0,187070.0,Cloître des Billettes,https://www.tripadvisor.fr/Attraction_Review-g...,2.942987,23,23.0,48.858000,2.354980,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


We can see that the _id_ column in this dataset corresponds to the _idplace_ column in the **reviews** dataset. Therefore, the **reviews** dataset is simply the extended version, with all the reviews relating to the lines in the **tripadvisor** dataset.

In [54]:
tripadvisor.shape

(3761, 60)

#### Keep only english reviews
In order to apply the data to a model, it is necessary to use a specific language to simplify matters. We use English because the majority of journals are in this language.

In [55]:
reviews = reviews[reviews.langue == 'en']
reviews.shape

(153071, 21)

#### Keep usefull features
Here for our project we just need to focus on reviews and the place, so we only keep the review and the idplace.

In [56]:
reviews = reviews[['idplace', 'review']]
reviews.head()

,idplace,review
0,188467,Personally I think it is the most beautiful sq...
1,188467,We walked through this lovely park but did not...
2,188467,We come back to this huge square every time we...
3,188467,The most beautiful square in Paris has a stran...
4,188467,"Lovely architecture, looks different again to ..."


In [57]:
tripadvisor = tripadvisor[['id','nom', 'rating','activiteSubType','activiteSubCategorie','restaurantTypeCuisine','restaurantDietaryRestrictions']]
tripadvisor.head()

,id,nom,rating,activiteSubType,activiteSubCategorie,restaurantTypeCuisine,restaurantDietaryRestrictions
0,188467,Place des Vosges,4.108407,163,47,NaN,NaN
1,188468,Rue des Francs Bourgeois,3.316532,163,47,NaN,NaN
2,188470,Village Saint-Paul,3.017118,"34,144","26,47,51",NaN,NaN
3,188471,Au Passe-partout,2.743157,137,26,NaN,NaN
4,188472,Cloître des Billettes,2.942987,10,47,NaN,NaN


In [58]:
tripadvisor.dtypes

id                                 int64
nom                               object
rating                           float64
activiteSubType                   object
activiteSubCategorie              object
restaurantTypeCuisine             object
restaurantDietaryRestrictions     object
dtype: object

We can see that the type columns do not have the correct format; we need to accept a list of integers containing the ID values.

In [59]:
tripadvisor['activiteSubType'] = tripadvisor['activiteSubType'].apply(
    lambda s: [int(x.strip()) for x in str(s).split(",") if x is not np.nan and x != 'nan']
)
tripadvisor['activiteSubCategorie'] = tripadvisor['activiteSubCategorie'].apply(
    lambda s: [int(x.strip()) for x in str(s).split(",") if x is not np.nan and x != 'nan']
)
tripadvisor['restaurantTypeCuisine'] = tripadvisor['restaurantTypeCuisine'].apply(
    lambda s: [int(x.strip()) for x in str(s).split(",") if x is not np.nan and x != 'nan']
)
tripadvisor['restaurantDietaryRestrictions'] = tripadvisor['restaurantDietaryRestrictions'].apply(
    lambda s: [int(x.strip()) for x in str(s).split(",") if x is not np.nan and x != 'nan']
)
tripadvisor.head()


,id,nom,rating,activiteSubType,activiteSubCategorie,restaurantTypeCuisine,restaurantDietaryRestrictions
0,188467,Place des Vosges,4.108407,[163],[47],[],[]
1,188468,Rue des Francs Bourgeois,3.316532,[163],[47],[],[]
2,188470,Village Saint-Paul,3.017118,"[34, 144]","[26, 47, 51]",[],[]
3,188471,Au Passe-partout,2.743157,[137],[26],[],[]
4,188472,Cloître des Billettes,2.942987,[10],[47],[],[]


### Load place dataset

In [60]:
attractionSubCat = pd.read_csv('data/AttractionSubCategorie.csv')
attractionSubType = pd.read_csv('data/AttractionSubType.csv')
cuisine = pd.read_csv('data/cuisine.csv')
dietaryRestrictions = pd.read_csv('data/dietary_restrictions.csv')

In [61]:
print(attractionSubCat.shape, attractionSubCat.head())
print(attractionSubType.shape, attractionSubType.head())
print(cuisine.shape, cuisine.head())
print(dietaryRestrictions.shape, dietaryRestrictions.head())

(20, 2)    id               name
0  20       Vie nocturne
1  26           Shopping
2  36       Restauration
3  40  Spas et bien-être
4  41  Cours et ateliers
(229, 2)    id                      name
0   1            Galeries d'art
1   2          Ruines anciennes
2   3  Bâtiments architecturaux
3   4        Champs de bataille
4   5                     Ponts
(161, 2)      id      name
0  4617   Italian
1  5086    French
2  5110   Mexican
3  5379   Chinese
4  5473  Japanese
(5, 2)       id                 name
0  10665  Vegetarian Friendly
1  10697        Vegan Options
2  10751                Halal
3  10768               Kosher
4  10992  Gluten Free Options


#### We will add a standard column to indicate the file name, e.g. AttractionSubCategory, and then be able to merge all these dataframes

In [62]:
attractionSubCat['type'] = 'AttractionSubCategorie'
attractionSubType['type'] = 'AttractionSubType'
cuisine['type'] = 'Cuisine'
dietaryRestrictions['type'] = 'DietaryRestriction'

col = ["type", "id", "name"]
infos_places = pd.concat(
    [attractionSubCat[col],
     attractionSubType[col],
     cuisine[col],
     dietaryRestrictions[col]],
    ignore_index=True
)

# Save it to a new CSV file
infos_places.to_csv("data/infos_places.csv", index=False)

print(infos_places.shape)
infos_places.head()

(415, 3)


,type,id,name
0,AttractionSubCategorie,20,Vie nocturne
1,AttractionSubCategorie,26,Shopping
2,AttractionSubCategorie,36,Restauration
3,AttractionSubCategorie,40,Spas et bien-être
4,AttractionSubCategorie,41,Cours et ateliers


## Basic text cleaning
1. Convert reviews to lowercase

In [63]:
reviews['review'] = reviews['review'].apply(lambda x: x.lower())
reviews.head()

,idplace,review
0,188467,personally i think it is the most beautiful sq...
1,188467,we walked through this lovely park but did not...
2,188467,we come back to this huge square every time we...
3,188467,the most beautiful square in paris has a stran...
4,188467,"lovely architecture, looks different again to ..."


2. Remove punctuation and numbers if necessary

In [64]:
reviews['review'].iloc[0]

'personally i think it is the most beautiful square of paris. well maintained and the area around it gives you opportunities to grab a bite to eat as well.'

In [65]:
#!pip3 install nltk
import ssl

# Permet de contourner les problèmes de certificat SSL lors du téléchargement des ressources NLTK
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    # Environnements qui n'ont pas cette méthode (vieux Python)
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

import nltk
nltk.download('punkt_tab') # tokenizers
nltk.download('stopwords') # listes de stopwords
nltk.download('wordnet')  # WordNet pour la lemmatisation
import string
from nltk.tokenize import TreebankWordTokenizer

[nltk_data] Downloading package punkt_tab to /Users/marya/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /Users/marya/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/marya/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [66]:
wordTokenizer = TreebankWordTokenizer()
reviews['review'] = reviews['review'].apply(lambda x: [mot for mot in wordTokenizer.tokenize(x) if mot not in string.punctuation])
reviews['review'].iloc[0]

['personally',
 'i',
 'think',
 'it',
 'is',
 'the',
 'most',
 'beautiful',
 'square',
 'of',
 'paris.',
 'well',
 'maintained',
 'and',
 'the',
 'area',
 'around',
 'it',
 'gives',
 'you',
 'opportunities',
 'to',
 'grab',
 'a',
 'bite',
 'to',
 'eat',
 'as',
 'well']

3. Remove stopwords

In [67]:
from nltk.corpus import stopwords

stop = stopwords.words('english')

reviews['review'] = reviews['review'].apply(lambda x: [mot for mot in x if mot not in stop])
reviews['review'].iloc[0]

['personally',
 'think',
 'beautiful',
 'square',
 'paris.',
 'well',
 'maintained',
 'area',
 'around',
 'gives',
 'opportunities',
 'grab',
 'bite',
 'eat',
 'well']

4. Lemmatisation

In [68]:
from nltk.stem import WordNetLemmatizer

lemmatizer_output = WordNetLemmatizer()

reviews['review'] = reviews['review'].apply(lambda x: [lemmatizer_output.lemmatize(mot) for mot in x])
reviews['review'].iloc[0]

['personally',
 'think',
 'beautiful',
 'square',
 'paris.',
 'well',
 'maintained',
 'area',
 'around',
 'give',
 'opportunity',
 'grab',
 'bite',
 'eat',
 'well']

**Comments** We don't see much change because the words are already in their lemmatised form, but we can see that some words are in the singular, such as give and opportunity.

## Reduction in the number of reviews per location

1. Group by idplace & for each one, calculate the length of each review (total token)

In [69]:
# group by idplace and concatenate the reviews
reviews = reviews.groupby('idplace').agg({'review': 'sum'})
reviews.head()

,review
idplace,
188467,"[personally, think, beautiful, square, paris.,..."
188468,"[old, college, friend, booked, beautiful, expe..."
188470,"[winter, lot, going, however, easy, see, cobbl..."
188471,"[call, au, passe, partout, shop, serious, unde..."
188472,"[old, historical, place., attended, experience..."


In [ ]:
# sort by number of tokens
reviews['review_length'] = reviews['review'].apply(lambda x: len(x))
reviews.head()

,review,review_length
idplace,,
188467,"[personally, think, beautiful, square, paris.,...",56902
188468,"[old, college, friend, booked, beautiful, expe...",694
188470,"[winter, lot, going, however, easy, see, cobbl...",2048
188471,"[call, au, passe, partout, shop, serious, unde...",54
188472,"[old, historical, place., attended, experience...",113


2. Retain as many of the most frequent N tokens per review

In [72]:
from collections import Counter

# by default K is the mean of the review lengths to keep a similar number of words in each review
K = int(reviews["review_length"].mean())

# sort the tokens by frequency and keep only the top K tokens
def top_k_tokens(tokens, k=None):
    if k is None:
        k = len(tokens)
    k = int(k)

    c = Counter(tokens)
    return [mot for mot, freq in c.most_common(k)]

reviews["top_tokens"] = reviews["review"].apply(
    lambda toks: top_k_tokens(toks, k=K)
)

reviews.head()



,review,review_length,top_tokens
idplace,,,
188467,"[personally, think, beautiful, square, paris.,...",56902,"[place, square, de, park, paris, beautiful, 's..."
188468,"[old, college, friend, booked, beautiful, expe...",694,"[street, one, apartment, shopping, lot, area, ..."
188470,"[winter, lot, going, however, easy, see, cobbl...",2048,"[shop, place, one, village, little, antique, a..."
188471,"[call, au, passe, partout, shop, serious, unde...",54,"[history, n't, owner, unique, call, au, passe,..."
188472,"[old, historical, place., attended, experience...",113,"[de, art, place, church, paris., billettes, cl..."


3. Create text_resume ready for TF‑IDF and create a ready-to-use dataframe

In [75]:
reviews["text_resume"] = reviews["top_tokens"].apply(lambda toks: " ".join(toks))
reviews.head()

,review,review_length,top_tokens,text_resume
idplace,,,,
188467,"[personally, think, beautiful, square, paris.,...",56902,"[place, square, de, park, paris, beautiful, 's...",place square de park paris beautiful 's one vo...
188468,"[old, college, friend, booked, beautiful, expe...",694,"[street, one, apartment, shopping, lot, area, ...",street one apartment shopping lot area marais ...
188470,"[winter, lot, going, however, easy, see, cobbl...",2048,"[shop, place, one, village, little, antique, a...",shop place one village little antique area pau...
188471,"[call, au, passe, partout, shop, serious, unde...",54,"[history, n't, owner, unique, call, au, passe,...",history n't owner unique call au passe partout...
188472,"[old, historical, place., attended, experience...",113,"[de, art, place, church, paris., billettes, cl...",de art place church paris. billettes cloister ...


In [84]:
reviews_for_tfidf = reviews[['text_resume']]
reviews_for_tfidf = reviews_for_tfidf.reset_index()

# save the preprocessed reviews to a new CSV file
reviews_for_tfidf.to_csv("data/preprocessed_reviews.csv", index=True)
reviews_for_tfidf.head()

,idplace,text_resume
0,188467,place square de park paris beautiful 's one vo...
1,188468,street one apartment shopping lot area marais ...
2,188470,shop place one village little antique area pau...
3,188471,history n't owner unique call au passe partout...
4,188472,de art place church paris. billettes cloister ...
